# PydanticAI together with Pydantic Models

## PydanticAI utan Pydantic-modell – format via prompt

Här beskrivs önskat format bara som text i `instructions`. Modellen följer oftast exemplet, men:
- `result.output` är en **sträng**, inte ett objekt. För att komma åt `name` eller `age` måste du parsa texten själv.
- Det finns **ingen garanti**. Modellen kan byta fältnamn, hoppa över fält eller skriva lön som "35 000 kr".
- Ingen validering, t.ex. att åldern är rimlig.

Det är problemet som structured output löser.

Obs: `instructions` består av två strängar efter varandra (`"..." """..."""`). Python slår automatiskt ihop dem (implicit string concatenation).

In [51]:
from model import nvidia_model
from pydantic_ai import Agent

employee_simulator = Agent(model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
instructions="Du är expert på att simulera anställda i Sverige. Simulera i detta formatet"
"""
    name: "Bertil",
    age: 40,
    salary: 35000,
    role: "IT-supporter
    ort: Stockholm
""")

result = await employee_simulator.run("Ge mig en data engineer ")
print(result.output)

name: "Sofia",
age: 34,
salary: 52000,
role: "Data Engineer",
ort: "Göteborg"


## Force a structured output using pydantic

## Structured output med Pydantic

`EmployeeModel` är ett **schema**: det beskriver exakt vilka fält som finns, vilken typ de har och vilka värden som är tillåtna.

- `BaseModel`: basklassen som ger validering, typkonvertering och serialisering.
- `Field(description=...)`: beskrivningen skickas till LLM:en som en del av JSON-schemat. Den fungerar alltså som en **instruktion per fält**.
- `ge` / `le` / `lt`: valideringsregler (greater-or-equal, less-or-equal, less-than). `age` måste vara 18–67 och `salary` 30 000–59 999.

`employee1` skapas manuellt och visar att modellen fungerar som en vanlig klass. Pydantic validerar redan när objektet skapas, så `age=12` skulle ge ett `ValidationError`.

In [ ]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str = Field(
        description="för och efternamn, inte jättelångt, svenska eller internationella namn"
    )
    age: int = Field(description="ålder från 18 till 67 år", ge=18, le=67)
    salary: int = Field(
        description="lön mellan 30k-60k, ju mer senior position, desto högre lön",
        ge=30_000,
        lt=60_000,
    )
    role: str = Field(description="yrke/roll")
    description: str = "kort beskrivning, 2 meningar på rollen"

employee1 = EmployeeModel(name="Bertil", age=40, salary= 50_000, role="It-supporter", description="Hjälper folk")

### Tvinga fram strukturen med `output_type`

`output_type=EmployeeModel` gör att PydanticAI:
1. skickar Pydantic-modellens JSON-schema till LLM:en
2. tar emot svaret som JSON
3. validerar det mot `EmployeeModel`. Om valideringen misslyckas skickas felet tillbaka till modellen, som får försöka igen.

Resultatet blir ett **riktigt Python-objekt**, inte text.

In [53]:
employee_simulator = Agent(model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
instructions="Du är expert på att simulera anställda i Sverige",
)

result = await employee_simulator.run("Ge mig en data engineer i Stockholm", output_type=EmployeeModel)

result

AgentRunResult(output=EmployeeModel(name='Erik Johansson', age=34, salary=52000, role='Data Engineer', description='Arbetar med att bygga och underhålla datapipelines i molnet för ett techbolag i Stockholm. Erfarenhet av Python, SQL och stora datamängder.'))

### Arbeta med resultatet som objekt

`result.output` är nu en `EmployeeModel`-instans:
- Fälten nås med punktnotation: `employee2.name`
- Typerna är garanterade: `employee2.age` är en `int`, så du kan räkna med den direkt.
- `__dict__` visar objektets attribut som en dict. Det Pydantic-idiomatiska sättet är `model_dump()`, som också hanterar nästlade modeller.

In [54]:
employee2 = result.output

In [55]:
employee2.name + " är cool"

'Erik Johansson är cool'

In [56]:
employee2.__dict__.keys()

dict_keys(['name', 'age', 'salary', 'role', 'description'])

In [57]:
employee2.__dict__.values()

dict_values(['Erik Johansson', 34, 52000, 'Data Engineer', 'Arbetar med att bygga och underhålla datapipelines i molnet för ett techbolag i Stockholm. Erfarenhet av Python, SQL och stora datamängder.'])

In [58]:
for key, value in employee2.__dict__.items():
    print(f"{key}: {value}")

name: Erik Johansson
age: 34
salary: 52000
role: Data Engineer
description: Arbetar med att bygga och underhålla datapipelines i molnet för ett techbolag i Stockholm. Erfarenhet av Python, SQL och stora datamängder.


# Simulate many employees

## Generera många objekt på en gång

`output_type=list[EmployeeModel]` ger en **lista** med validerade objekt. Varje element kontrolleras mot schemat.

Eftersom alla löner är `int` kan vi räkna fram ett medelvärde direkt, utan att parsa text.

In [59]:
result = await employee_simulator.run("Ge mig totalt tio anställda inom IT/Data/Sälj",
                                      output_type=list[EmployeeModel])

result

AgentRunResult(output=[EmployeeModel(name='Erik Andersson', age=34, salary=48000, role='Systemutvecklare', description='Utvecklar webbapplikationer i .NET och Azure. Har fem års erfarenhet av konsultbranschen.'), EmployeeModel(name='Sofia Karlsson', age=29, salary=42000, role='Dataanalytiker', description='Arbetar med datavisualisering i Power BI och SQL. Stödjer affärsområden med beslutsunderlag.'), EmployeeModel(name='Johan Lindberg', age=41, salary=56000, role='IT-projektledare', description='Ledar stora IT-transformationsprojekt i detaljhandeln. Certifierad PMP och Scrum Master.'), EmployeeModel(name='Maria Bergström', age=38, salary=51000, role='Säljchef', description='Ansvarar för ett säljteam inom B2B-mjukvara. Fokuserar på kundrelationer och intäktstillväxt.'), EmployeeModel(name='Amir Hassan', age=26, salary=38000, role='DevOps-ingenjör', description='Hanterar CI/CD-pipelines och molninfrastruktur i AWS. Brinner för automatisering och driftssäkerhet.'), EmployeeModel(name='Lis

In [ ]:
employees = result.output

salary = 0
for employee in employees:
    salary = employee.salary

salary/len(employees)


5800.0

# Convert to dataframe

## Från Pydantic till DataFrame och CSV

- `model_dump()` gör om ett Pydantic-objekt till en vanlig dict. Det kallas **serialisering**.
- En lista med dicts blir en DataFrame direkt, där varje dict blir en rad och nycklarna blir kolumner.
- `to_csv()` sparar till fil, så att datan kan användas vidare (Power BI, databas, nästa pipeline-steg).

Kedjan: **LLM → JSON → validerad Pydantic-modell → dict → DataFrame → CSV**

In [61]:
employees[0].model_dump()

{'name': 'Erik Andersson',
 'age': 34,
 'salary': 48000,
 'role': 'Systemutvecklare',
 'description': 'Utvecklar webbapplikationer i .NET och Azure. Har fem års erfarenhet av konsultbranschen.'}

In [62]:
import pandas as pd
df = pd.DataFrame([employee.model_dump() for employee in employees])
df.head()

,name,age,salary,role,description
0,Erik Andersson,34,48000,Systemutvecklare,Utvecklar webbapplikationer i .NET och Azure. ...
1,Sofia Karlsson,29,42000,Dataanalytiker,Arbetar med datavisualisering i Power BI och S...
2,Johan Lindberg,41,56000,IT-projektledare,Ledar stora IT-transformationsprojekt i detalj...
3,Maria Bergström,38,51000,Säljchef,Ansvarar för ett säljteam inom B2B-mjukvara. F...
4,Amir Hassan,26,38000,DevOps-ingenjör,Hanterar CI/CD-pipelines och molninfrastruktur...


In [63]:
df.to_csv("employees.csv", index=None)